In [2]:
# ==========================================
# Notebook 09
# Cold Start Recommendation Engine
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [4]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [5]:
index = faiss.read_index("../data/item_index.faiss")

index.ntotal

0

In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [7]:
new_item = {
    "item_id": 999,
    "title": "OpenAI Startup Journey",
    "category": "Artificial Intelligence",
    "description": """
    How OpenAI evolved from a
    research organization into a
    leading artificial intelligence company.
    """,
}

In [8]:
new_item_profile = f"""

Title: {new_item['title']}

Category: {new_item['category']}

Description: {new_item['description']}

"""

In [9]:
print(new_item_profile)



Title: OpenAI Startup Journey

Category: Artificial Intelligence

Description: 
    How OpenAI evolved from a
    research organization into a
    leading artificial intelligence company.
    




In [10]:
new_item_embedding = embedding_model.encode(new_item_profile)

In [11]:
new_item_embedding.shape

(384,)

In [12]:
similarities = cosine_similarity(new_item_embedding.reshape(1, -1), item_embeddings)[0]

In [16]:
similarities

array([0.3236242 , 0.43989807, 0.3812979 , 0.2052139 , 0.19153471,
       0.29378378, 0.35556078, 0.37782097, 0.24789613, 0.32044888],
      dtype=float32)

In [14]:
ranked_idx = np.argsort(similarities)[::-1]

In [17]:
ranked_idx[:10]

array([1, 2, 7, 6, 0, 9, 5, 8, 3, 4], dtype=int64)

In [18]:
results = []

for idx in ranked_idx[:5]:

    results.append(
        {
            "item_id": profiles_df.iloc[idx]["item_id"],
            "title": profiles_df.iloc[idx]["title"],
            "category": profiles_df.iloc[idx]["category"],
            "similarity": similarities[idx],
        }
    )

pd.DataFrame(results)

,item_id,title,category,similarity
0,2,Building SpaceX,Startup,0.439898
1,3,AI for Healthcare,Artificial Intelligence,0.381298
2,8,Startup Fundraising Guide,Startup,0.377821
3,7,Machine Learning in Finance,Machine Learning,0.355561
4,1,The Early Days of Stripe,Startup,0.323624


In [19]:
distances, indices = index.search(
    new_item_embedding.reshape(1, -1).astype("float32"), 5
)

In [20]:
indices

array([[-1, -1, -1, -1, -1]], dtype=int64)

In [21]:
cold_start_results = []

for rank, idx in enumerate(indices[0]):

    cold_start_results.append(
        {
            "rank": rank + 1,
            "item_id": profiles_df.iloc[idx]["item_id"],
            "title": profiles_df.iloc[idx]["title"],
            "category": profiles_df.iloc[idx]["category"],
            "distance": distances[0][rank],
        }
    )

cold_start_df = pd.DataFrame(cold_start_results)

cold_start_df

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [22]:
user_embeddings = np.load("../data/user_embeddings.npy")

user_profiles_df = pd.read_csv("../data/user_profiles.csv")

In [23]:
user_scores = cosine_similarity(new_item_embedding.reshape(1, -1), user_embeddings)[0]

In [24]:
user_scores

array([0.41534603, 0.42996803, 0.25455403, 0.42181644, 0.36963978],
      dtype=float32)

In [25]:
ranked_users = np.argsort(user_scores)[::-1]

In [26]:
user_recommendations = []

for idx in ranked_users:

    user_recommendations.append(
        {
            "user_id": user_profiles_df.iloc[idx]["user_id"],
            "similarity": user_scores[idx],
        }
    )

pd.DataFrame(user_recommendations)

,user_id,similarity
0,102.0,0.429968
1,104.0,0.421816
2,101.0,0.415346
3,105.0,0.369640
4,103.0,0.254554


In [27]:
def recommend_new_item_to_users(item_profile, top_k_users=5):

    item_embedding = embedding_model.encode(item_profile)

    scores = cosine_similarity(item_embedding.reshape(1, -1), user_embeddings)[0]

    ranked_users = np.argsort(scores)[::-1]

    recommendations = []

    for idx in ranked_users[:top_k_users]:

        recommendations.append(
            {"user_id": user_profiles_df.iloc[idx]["user_id"], "score": scores[idx]}
        )

    return pd.DataFrame(recommendations)

In [28]:
recommend_new_item_to_users(new_item_profile)

,user_id,score
0,102.0,0.429968
1,104.0,0.421816
2,101.0,0.415346
3,105.0,0.369640
4,103.0,0.254554


In [29]:
new_items = [
    """
    AI startup building
    next generation language models
    """,
    """
    Healthy nutrition and wellness
    strategies for long term fitness
    """,
    """
    Cloud computing architecture
    and distributed systems
    """,
]

In [30]:
for item in new_items:

    print()

    print("=" * 60)

    print(item)

    print("=" * 60)

    print()

    display(recommend_new_item_to_users(item))



    AI startup building
    next generation language models
    



,user_id,score
0,104.0,0.424704
1,102.0,0.289442
2,101.0,0.265551
3,105.0,0.229642
4,103.0,-0.026578




    Healthy nutrition and wellness
    strategies for long term fitness
    



,user_id,score
0,103.0,0.490902
1,102.0,0.050355
2,104.0,0.039230
3,101.0,-0.004896
4,105.0,-0.007924




    Cloud computing architecture
    and distributed systems
    



,user_id,score
0,105.0,0.406760
1,101.0,0.325800
2,104.0,0.155506
3,102.0,0.031689
4,103.0,-0.015555


In [31]:
cold_start_df.to_csv("../data/cold_start_item_matches.csv", index=False)

In [32]:
user_target_df = pd.DataFrame(user_recommendations)

user_target_df.to_csv("../data/cold_start_user_targets.csv", index=False)

In [33]:
saved_df = pd.read_csv("../data/cold_start_item_matches.csv")

saved_df.head()

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38
